# Embedding Extraction from Finetuned LLMs

This notebook extracts hidden state embeddings from finetuned LLMs for regression analysis.

## Purpose
Test whether embeddings from finetuned LLMs encode material property information that can be recovered via linear regression.

## Workflow
1. Load finetuned model (base + LoRA adapter)
2. Run inference and extract hidden states at each layer
3. Train Ridge regression on embeddings
4. Compare embedding regression performance to direct LLM prediction

## 1. Setup

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configuration
DATA_DIR = '../data'
RESULTS_DIR = '../results/embeddings'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Models and their HuggingFace paths
BASE_MODELS = {
    'llama-2-7b-chat': 'meta-llama/Llama-2-7b-chat-hf',
    'llama-3-8b-instruct': 'meta-llama/Meta-Llama-3-8B-Instruct',
    'mistral-7b-instruct': 'mistralai/Mistral-7B-Instruct-v0.2'
}

# Adapter paths on HuggingFace (vinven7 namespace)
ADAPTERS = {
    'Bandgap': {
        'llama-2-7b-chat': 'vinven7/llama-2-7b-chat-Bandgap-finetuning-1',
        'llama-3-8b-instruct': 'vinven7/llama-3-8b-instruct-Bandgap-finetuning-1',
        'mistral-7b-instruct': 'vinven7/mistral-7b-instruct-v0-2-Bandgap-finetuning-4'
    },
    'Dielectric': {
        'llama-2-7b-chat': 'vinven7/llama-2-7b-chat-Dielectric-finetuning-2',
        'llama-3-8b-instruct': 'vinven7/llama-3-8b-instruct-Dielectric-finetuning-1',
        'mistral-7b-instruct': 'vinven7/mistral-7b-instruct-v0-2-Dielectric-finetuning-1'
    }
}

# Tasks for embedding analysis
TASKS = ['Bandgap', 'Dielectric']

## 2. Model Loading

In [ ]:
def load_model_with_adapter(base_model_name, adapter_path, use_4bit=True):
    """
    Load base model with LoRA adapter.
    
    Args:
        base_model_name: HuggingFace model name
        adapter_path: Path to LoRA adapter (HF or local)
        use_4bit: Whether to use 4-bit quantization
    
    Returns:
        model, tokenizer
    """
    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Quantization config for memory efficiency
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            quantization_config=bnb_config,
            device_map="auto"
        )
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.float16,
            device_map="auto"
        )
    
    # Load adapter
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    
    return model, tokenizer

## 3. Data Loading & Prompt Processing

In [ ]:
def load_data(task, model_name, split='Train'):
    """Load data for a task-model combination."""
    filepath = os.path.join(DATA_DIR, task, f'{task}_{model_name}_{split}.csv')
    return pd.read_csv(filepath)

def parse_list_column(val):
    """Parse string list representation."""
    import ast
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except:
            return [val]
    return [val]

def create_single_formula_prompt(original_prompt, formula):
    """
    Create a prompt for a single formula from a batched prompt template.
    
    The batched prompt contains multiple formulas. We replace the list
    with a single formula for embedding extraction.
    """
    # Replace the materials list with single formula
    pattern = r"(MATERIALS:\s*)\[[^\]]*\]"
    return re.sub(pattern, f"\\g<1>{formula}", original_prompt)

def unbatch_dataframe(df):
    """
    Unbatch a dataframe where each row contains lists of samples.
    
    Returns a dataframe with one row per sample.
    """
    rows = []
    for _, row in df.iterrows():
        completions = parse_list_column(row['completion'])
        formulas = parse_list_column(row['Formula'])
        mpids = parse_list_column(row.get('mpid', [None] * len(completions)))
        prompt_template = row['prompt']
        
        for i in range(len(completions)):
            rows.append({
                'completion': completions[i],
                'Formula': formulas[i],
                'mpid': mpids[i] if i < len(mpids) else None,
                'prompt_template': prompt_template
            })
    
    return pd.DataFrame(rows)

## 4. Embedding Extraction

In [ ]:
def extract_embeddings(model, tokenizer, prompts, layers=None, batch_size=1):
    """
    Extract hidden state embeddings from model.
    
    Args:
        model: The model
        tokenizer: The tokenizer
        prompts: List of prompts
        layers: List of layer indices to extract (None = all layers)
        batch_size: Batch size for processing
    
    Returns:
        Dict mapping layer index to list of embeddings
    """
    embeddings_by_layer = {}
    
    model.eval()
    
    for i in tqdm(range(0, len(prompts), batch_size), desc="Extracting embeddings"):
        batch_prompts = prompts[i:i+batch_size]
        
        # Tokenize
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(device)
        
        with torch.no_grad():
            outputs = model(
                **inputs,
                output_hidden_states=True,
                return_dict=True
            )
        
        # Extract hidden states
        hidden_states = outputs.hidden_states  # Tuple of (n_layers+1, batch, seq_len, hidden_dim)
        
        # Determine which layers to extract
        if layers is None:
            layers_to_extract = range(len(hidden_states))
        else:
            layers_to_extract = layers
        
        for layer_idx in layers_to_extract:
            if layer_idx not in embeddings_by_layer:
                embeddings_by_layer[layer_idx] = []
            
            # Get last token embedding (most common approach for causal LMs)
            layer_hidden = hidden_states[layer_idx]  # (batch, seq_len, hidden_dim)
            
            # Get last non-padding token for each sample
            for j in range(len(batch_prompts)):
                # Find last non-padding position
                attention_mask = inputs['attention_mask'][j]
                last_pos = attention_mask.sum().item() - 1
                
                embedding = layer_hidden[j, last_pos, :].cpu()
                embeddings_by_layer[layer_idx].append(embedding)
    
    return embeddings_by_layer

In [ ]:
def extract_embeddings_for_task(task, model_name, split='Train'):
    """
    Extract embeddings for all samples in a task dataset.
    
    Args:
        task: Task name ('Bandgap' or 'Dielectric')
        model_name: Short model name
        split: 'Train' or 'Test'
    
    Returns:
        Dict with embeddings and metadata
    """
    # Load data
    df = load_data(task, model_name, split)
    df_unbatched = unbatch_dataframe(df)
    
    print(f"Loaded {len(df_unbatched)} samples")
    
    # Load model
    base_model_path = BASE_MODELS[model_name]
    adapter_path = ADAPTERS[task][model_name]
    
    print(f"Loading model: {base_model_path}")
    print(f"Loading adapter: {adapter_path}")
    
    model, tokenizer = load_model_with_adapter(base_model_path, adapter_path)
    
    # Create single-formula prompts
    prompts = [
        create_single_formula_prompt(row['prompt_template'], row['Formula'])
        for _, row in df_unbatched.iterrows()
    ]
    
    # Extract embeddings (focus on later layers)
    n_layers = model.config.num_hidden_layers
    layers_to_extract = list(range(n_layers // 2, n_layers + 1))  # Second half + final
    
    embeddings_by_layer = extract_embeddings(
        model, tokenizer, prompts,
        layers=layers_to_extract,
        batch_size=1
    )
    
    # Package results
    results = {}
    for layer_idx, embeddings in embeddings_by_layer.items():
        results[layer_idx] = []
        for i, emb in enumerate(embeddings):
            results[layer_idx].append({
                'embedding': emb,
                'response': float(df_unbatched.iloc[i]['completion']),
                'formula': df_unbatched.iloc[i]['Formula'],
                'mpid': df_unbatched.iloc[i]['mpid']
            })
    
    # Cleanup
    del model
    torch.cuda.empty_cache()
    
    return results

## 5. Regression Analysis

In [ ]:
def train_and_evaluate_regression(train_data, test_data, layer):
    """
    Train Ridge regression on embeddings and evaluate.
    
    Args:
        train_data: Training embeddings dict
        test_data: Test embeddings dict
        layer: Layer index to use
    
    Returns:
        Dict with metrics
    """
    # Prepare data
    X_train = np.vstack([s['embedding'].numpy().flatten() for s in train_data[layer]])
    y_train = np.array([s['response'] for s in train_data[layer]])
    
    X_test = np.vstack([s['embedding'].numpy().flatten() for s in test_data[layer]])
    y_test = np.array([s['response'] for s in test_data[layer]])
    
    # Standardize
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    # Train Ridge regression
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_s, y_train)
    
    # Predict
    y_pred = ridge.predict(X_test_s)
    
    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    return {
        'layer': layer,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'n_train': len(y_train),
        'n_test': len(y_test)
    }

def analyze_all_layers(train_data, test_data):
    """
    Run regression analysis for all available layers.
    """
    results = []
    layers = sorted(train_data.keys())
    
    for layer in tqdm(layers, desc="Analyzing layers"):
        metrics = train_and_evaluate_regression(train_data, test_data, layer)
        results.append(metrics)
    
    return pd.DataFrame(results)

## 6. Run Full Pipeline

In [ ]:
def run_embedding_analysis(task, model_name):
    """
    Run complete embedding extraction and regression analysis.
    
    Args:
        task: Task name
        model_name: Model short name
    """
    output_dir = os.path.join(RESULTS_DIR, f"{task}_{model_name}")
    os.makedirs(output_dir, exist_ok=True)
    
    # Check if already done
    if os.path.exists(os.path.join(output_dir, 'regression_results.csv')):
        print(f"Skipping (exists): {output_dir}")
        return
    
    print(f"\n{'='*60}")
    print(f"Task: {task} | Model: {model_name}")
    print('='*60)
    
    # Extract train embeddings
    print("\nExtracting train embeddings...")
    train_data = extract_embeddings_for_task(task, model_name, 'Train')
    torch.save(train_data, os.path.join(output_dir, 'train_embeddings.pt'))
    
    # Extract test embeddings
    print("\nExtracting test embeddings...")
    test_data = extract_embeddings_for_task(task, model_name, 'Test')
    torch.save(test_data, os.path.join(output_dir, 'test_embeddings.pt'))
    
    # Run regression analysis
    print("\nRunning regression analysis...")
    results_df = analyze_all_layers(train_data, test_data)
    results_df.to_csv(os.path.join(output_dir, 'regression_results.csv'), index=False)
    
    # Print summary
    best_layer = results_df.loc[results_df['rmse'].idxmin()]
    print(f"\nBest layer: {int(best_layer['layer'])}")
    print(f"  RMSE: {best_layer['rmse']:.4f}")
    print(f"  MAE:  {best_layer['mae']:.4f}")
    print(f"  R²:   {best_layer['r2']:.4f}")
    
    return results_df

In [ ]:
# Example: Run for single task-model
# Uncomment to run

# results = run_embedding_analysis('Bandgap', 'llama-3-8b-instruct')

In [ ]:
# Run for all task-model combinations
# Uncomment to run all

# for task in TASKS:
#     for model_name in BASE_MODELS.keys():
#         if model_name in ADAPTERS.get(task, {}):
#             run_embedding_analysis(task, model_name)

## 7. Compare Embedding Regression vs LLM Performance

In [ ]:
def compare_embedding_vs_llm(task, model_name):
    """
    Compare embedding regression performance to direct LLM prediction.
    
    Loads:
    - Embedding regression results
    - LLM prediction results (from summary CSV)
    """
    # Load embedding regression results
    emb_results_path = os.path.join(RESULTS_DIR, f"{task}_{model_name}", 'regression_results.csv')
    emb_results = pd.read_csv(emb_results_path)
    best_emb = emb_results.loc[emb_results['rmse'].idxmin()]
    
    # Load LLM results
    llm_results_path = f'../results/{task}_{model_name}_ft_{task}_summary.csv'
    if os.path.exists(llm_results_path):
        llm_df = pd.read_csv(llm_results_path)
        llm_rmse = np.sqrt(mean_squared_error(
            llm_df['Ground'], 
            llm_df['mode_prediction'].fillna(llm_df['Ground'].mean())
        ))
    else:
        llm_rmse = None
    
    comparison = {
        'task': task,
        'model': model_name,
        'embedding_rmse': best_emb['rmse'],
        'embedding_r2': best_emb['r2'],
        'best_layer': int(best_emb['layer']),
        'llm_rmse': llm_rmse,
        'gap_ratio': llm_rmse / best_emb['rmse'] if llm_rmse else None
    }
    
    return comparison

# Example usage
# comparison = compare_embedding_vs_llm('Bandgap', 'llama-3-8b-instruct')
# print(comparison)

## 8. Visualize Layer-wise Performance

In [ ]:
import matplotlib.pyplot as plt

def plot_layer_performance(task, model_name):
    """
    Plot regression performance across layers.
    """
    results_path = os.path.join(RESULTS_DIR, f"{task}_{model_name}", 'regression_results.csv')
    df = pd.read_csv(results_path)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # RMSE
    axes[0].plot(df['layer'], df['rmse'], 'o-')
    axes[0].set_xlabel('Layer')
    axes[0].set_ylabel('RMSE')
    axes[0].set_title(f'{task} - RMSE by Layer')
    axes[0].axhline(df['rmse'].min(), color='red', linestyle='--', alpha=0.5)
    
    # R²
    axes[1].plot(df['layer'], df['r2'], 'o-', color='green')
    axes[1].set_xlabel('Layer')
    axes[1].set_ylabel('R²')
    axes[1].set_title(f'{task} - R² by Layer')
    
    # MAE
    axes[2].plot(df['layer'], df['mae'], 'o-', color='orange')
    axes[2].set_xlabel('Layer')
    axes[2].set_ylabel('MAE')
    axes[2].set_title(f'{task} - MAE by Layer')
    
    plt.suptitle(f'{model_name}', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"{task}_{model_name}", 'layer_performance.png'), dpi=150)
    plt.show()

# Example usage
# plot_layer_performance('Bandgap', 'llama-3-8b-instruct')

## Notes

### Key Findings (from experiments)
- **Bandgap**: Embedding regression achieves ~1x LLM performance (embeddings encode property well)
- **Dielectric**: Embedding regression achieves ~2.7x worse than LLM (information encoded non-linearly or in generation process)

### Best Practices
1. Use later layers (typically layer 24+ for 32-layer models)
2. Extract embedding from last token position
3. StandardScaler improves regression performance

### Memory Optimization
- Use 4-bit quantization for large models
- Process in small batches
- Clear GPU cache between models